In [16]:
import os
from git import Repo

from langchain_text_splitters import RecursiveCharacterTextSplitter,Language
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.chains import ConversationalRetrievalChain

In [17]:
#cloning the repository
!mkdir -p test-repo

A subdirectory or file -p already exists.
Error occurred while processing: -p.
A subdirectory or file test-repo already exists.
Error occurred while processing: test-repo.


In [12]:
repo_path = "test-repo"
repo = Repo.clone_from("https://github.com/sauravyadav7721/Books-Recommender-System",to_path=repo_path)

In [13]:
%pwd

'c:\\Users\\Saurav Yadav\\Desktop\\Gen AI Project\\Realtime-Source-Code-Analyzer\\research'

In [18]:
loader = GenericLoader.from_filesystem(
    repo_path,
    glob="**/*.py",
    suffixes=[".py"],
    parser=LanguageParser(language=Language.PYTHON,parser_threshold=500)
)

In [19]:
documents = loader.load()

In [20]:
documents

[Document(metadata={'source': 'test-repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='import os\nimport sys\nimport pickle\nimport streamlit as st\nimport numpy as np\nfrom Books_Recommender_System.logger.log import logging\nfrom Books_Recommender_System.config.configuration import AppConfiguration\nfrom Books_Recommender_System.pipeline.pipeline import TrainingPipeline\nfrom Books_Recommender_System.exceptions.exception_handler import AppException\n\n\n\nclass Recommendation:\n    def __init__(self,app_config = AppConfiguration()):\n        try:\n            self.recommendation_config= app_config.get_recommendation_config()\n        except Exception as e:\n            raise AppException(e, sys) from e\n        \n    \n\n    def fetch_poster(self,suggestion):\n        try:\n            book_name = []\n            ids_index = []\n            poster_url = []\n            book_pivot =  pickle.load(open(self.recommendation_config.book_pivot_serialized_objects,\'rb\'))\

In [21]:
len(documents)

23

In [22]:
documents[0]

Document(metadata={'source': 'test-repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='import os\nimport sys\nimport pickle\nimport streamlit as st\nimport numpy as np\nfrom Books_Recommender_System.logger.log import logging\nfrom Books_Recommender_System.config.configuration import AppConfiguration\nfrom Books_Recommender_System.pipeline.pipeline import TrainingPipeline\nfrom Books_Recommender_System.exceptions.exception_handler import AppException\n\n\n\nclass Recommendation:\n    def __init__(self,app_config = AppConfiguration()):\n        try:\n            self.recommendation_config= app_config.get_recommendation_config()\n        except Exception as e:\n            raise AppException(e, sys) from e\n        \n    \n\n    def fetch_poster(self,suggestion):\n        try:\n            book_name = []\n            ids_index = []\n            poster_url = []\n            book_pivot =  pickle.load(open(self.recommendation_config.book_pivot_serialized_objects,\'rb\'))\n

In [23]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON,chunk_size=500,chunk_overlap=50)

In [24]:
texts = documents_splitter.split_documents(documents)

In [25]:
texts

[Document(metadata={'source': 'test-repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='import os\nimport sys\nimport pickle\nimport streamlit as st\nimport numpy as np\nfrom Books_Recommender_System.logger.log import logging\nfrom Books_Recommender_System.config.configuration import AppConfiguration\nfrom Books_Recommender_System.pipeline.pipeline import TrainingPipeline\nfrom Books_Recommender_System.exceptions.exception_handler import AppException'),
 Document(metadata={'source': 'test-repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='class Recommendation:\n    def __init__(self,app_config = AppConfiguration()):\n        try:\n            self.recommendation_config= app_config.get_recommendation_config()\n        except Exception as e:\n            raise AppException(e, sys) from e'),
 Document(metadata={'source': 'test-repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content="def fetch_poster(self,suggestion):\n        try:\n    

In [26]:
len(texts)

88

In [27]:
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

In [30]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\Saurav Yadav\.conda\envs\code_analyzer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2682.38it/s]


In [31]:
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="./data"
)

In [34]:
llm = ChatGroq(model = "openai/gpt-oss-120b",
    api_key=GROQ_API_KEY)

In [35]:
memeory = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)

C:\Users\Saurav Yadav\AppData\Local\Temp\ipykernel_10896\868129177.py:1: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memeory = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)


In [37]:
qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectordb.as_retriever(search_type="mmr",search_kwargs={"k":8}),
    memory=memeory
)

In [38]:
question = "What is the purpose of the code in this repository?"

In [39]:
result = qa({"question": question})

C:\Users\Saurav Yadav\AppData\Local\Temp\ipykernel_10896\1485584900.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa({"question": question})


In [40]:
print(result['answer'])

The repository implements a **Books Recommender System**.  

Its main purpose is to:

1. **Download and prepare a book‑rating dataset** – the code fetches a zip file from a URL, extracts it, and loads the data into pandas structures.  
2. **Validate and transform the data** – a data‑validation component checks the raw input, builds a user‑item rating matrix (`book_pivot`), and creates the objects that will be used later in the recommendation pipeline.  
3. **Train a recommendation model** – configuration classes such as `ModelTrainerConfig` and `ModelRecommendationConfig` (mentioned in the imports) define the parameters for model training and for generating recommendations.  
4. **Generate book recommendations** – given a user’s preferences, the pipeline computes a list of suggested books, gathers their titles, and fetches the corresponding poster images (via `fetch_poster`).  
5. **Persist results for later use** – the final rating matrix (or other intermediate objects) is serialized 

In [41]:
question = "Which machine learning code is used in this repository?"

In [42]:
result = qa({"question": question})

In [43]:
print(result['answer'])

The repository implements a **collaborative‑filtering recommender** that is built on top of the classic k‑Nearest‑Neighbors algorithm from **scikit‑learn**.

Here’s the key ML‑related code you can see in the snippets:

| Component | Library / Class | Purpose |
|-----------|----------------|---------|
| **ModelTrainer** | `from sklearn.neighbors import NearestNeighbors` | Trains a K‑NN model on the user‑item interaction matrix. |
| **Data representation** | `from scipy.sparse import csr_matrix` | Stores the (usually very sparse) book‑rating matrix efficiently for the K‑NN algorithm. |
| **Recommendation logic** (inside the `Recommendation` class) | Uses the trained `NearestNeighbors` model to find books that are “close” to the ones a user has liked, i.e., classic **item‑based collaborative filtering**. |

In short, the machine‑learning code in this repository is a **scikit‑learn NearestNeighbors‑based collaborative filtering system** (often referred to as a “k‑NN recommender”). It lever